# STALL — Inference Demo

**STALL** (Spatial-Temporal Anomaly Log-Likelihood) detects AI-generated videos by measuring how anomalous a video's spatial and temporal statistics are relative to a calibration set of real videos.

**Score interpretation:** `final_score ∈ [0, 1]` — **higher means more likely real**.

### Prerequisites
1. Clone DINOv3 and download weights:
```bash
git clone https://github.com/facebookresearch/dinov3 dinov3
mkdir -p dinov3/weights
# Download dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth into dinov3/weights/
```
2. Install dependencies: `pip install -r requirements.txt`

## 1. Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# Notebook lives in notebooks/ — repo root is one level up
REPO_ROOT = Path(os.path.abspath(".."))
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch

DEMO_VIDEOS_DIR = REPO_ROOT / "datasets" / "demo_dataset"
STALL_PARAMS    = REPO_ROOT / "precomputed" / "stall_params_vatex_dino_v3.npz"

# DINOv3 paths — edit if yours differ, or set env vars DINO_V3_REPO_DIR / DINO_V3_WEIGHTS
DINO_REPO    = Path(os.getenv("DINO_V3_REPO_DIR", str(REPO_ROOT / "dinov3")))
DINO_WEIGHTS = Path(os.getenv("DINO_V3_WEIGHTS",  str(DINO_REPO / "weights" /
                              "dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth")))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device:       {device}")
print(f"DINOv3 repo:  {DINO_REPO}")
print(f"DINOv3 weights exists: {DINO_WEIGHTS.exists()}")

## 2. Load STALL Model

In [ ]:
from stall import STALL

stall_params = np.load(STALL_PARAMS)
model = STALL(
    device=device,
    data_dict=stall_params,
    dino_repo=str(DINO_REPO),
    dino_weights=str(DINO_WEIGHTS),
)
print("STALL model ready.")

## 3. Single-Video Inference

In this example we use all frames as-is: `inference()` is called without `frame_indices`, so `load_video_frames()` in `src/stall.py` reads every frame at the video's native fps with no downsampling (see `stall.py` — `inference` and `load_video_frames`).

Run STALL on one real and one AI-generated video to see the raw output dictionary.

In [ ]:
# Pick one real and one fake video from the demo set
real_videos = sorted((DEMO_VIDEOS_DIR / "real").rglob("*.mp4"))
fake_videos = sorted((DEMO_VIDEOS_DIR / "fake").rglob("*.mp4"))

real_video = real_videos[0]
fake_video = fake_videos[0]

res_real = model.inference(str(real_video))
res_fake = model.inference(str(fake_video))

for label, res in [(f"Real  ({real_video.parent.name}/{real_video.name})", res_real),
                   (f"Fake  ({fake_video.parent.name}/{fake_video.name})", res_fake)]:
    print(f"\n{'─'*50}")
    print(f"  {label}")
    print(f"{'─'*50}")
    print(f"  Spatial  log-likelihood (max):  {res['spat_ll_agg'].item():.2f}")
    print(f"  Temporal log-likelihood (min):  {res['temp_ll_agg'].item():.2f}")
    print(f"  Spatial  percentile:            {res['spat_percentile'].item():.3f}")
    print(f"  Temporal percentile:            {res['temp_percentile'].item():.3f}")
    print(f"  ► Final score  (↑ = more real): {res['final_score'].item():.3f}")

### Optional: Per-Step Debug Output

`model.print_score_debug(result)` prints intermediate values for any inference result,
useful for verifying that the pipeline is behaving correctly (e.g. negative log-likelihoods,
unit whitened variance, real percentile > 0.5).

In [ ]:
SEP = "=" * 60
for label, res, path in [("REAL", res_real, real_video), ("FAKE", res_fake, fake_video)]:
    print(f"\n{SEP}")
    print(f"  {label}: {path.parent.name}/{path.name}")
    print(SEP)
    model.print_score_debug(res)
print(f"\n{SEP}")
print("  Expected: real final_score > fake final_score")
print(SEP)

## 4. Batch Inference on All Demo Videos

Discover and score every video in `datasets/demo_dataset/real/` and `datasets/demo_dataset/fake/`.

In [ ]:
records = []

for dir_path, subset_val in [(DEMO_VIDEOS_DIR / "real", "real"),
                              (DEMO_VIDEOS_DIR / "fake", "annotated")]:
    for model_dir in sorted(dir_path.iterdir()):
        if not model_dir.is_dir():
            continue
        for video_file in sorted(model_dir.glob("*.mp4")):
            res = model.inference(str(video_file))
            records.append({
                "video":        video_file.name,
                "source_model": model_dir.name,
                "subset":       subset_val,
                "spat_pct":     res["spat_percentile"].item(),
                "temp_pct":     res["temp_percentile"].item(),
                "final_score":  res["final_score"].item(),
            })
            print(f"[{subset_val:10s}] {model_dir.name:22s}  score={res['final_score'].item():.3f}")

df = pd.DataFrame(records)
print(f"\nDone — scored {len(df)} videos.")

## 5. Results Table

Sorted by `final_score` — real videos should appear near the top (higher score = more likely real).

In [ ]:
display(
    df.sort_values("final_score", ascending=False)
      .style
      .format({"spat_pct": "{:.3f}", "temp_pct": "{:.3f}", "final_score": "{:.3f}"})
      .background_gradient(subset=["final_score"], cmap="RdYlGn", vmin=0, vmax=1)
      .set_caption("STALL scores — higher final_score = more likely REAL")
)

## 6. AUC / AP on Demo Set

Compute detection metrics on the demo videos using the `metrics.py` utilities.

> **Note:** The demo set is small — these numbers are illustrative, not representative of paper results.

In [ ]:
from metrics import Score, ScoreDirection, get_results_df, print_results

scores_d = {
    "STALL": Score(
        value=df["final_score"].to_numpy(),
        direction=ScoreDirection.HIGHER_IS_REAL,
    )
}

results_df = get_results_df(df[["subset", "source_model"]], scores_d)
print_results(results_df)

## 7. Paper-Style Scoring with 2-Second Windows (8 fps Downsampling)

The inference above uses all frames at the video's native fps. The paper setup differs: videos are downsampled to 8 fps and scored over a randomly sampled 2-second window. To replicate this, use `video_index.py` to pre-compute frame indices (`2_sec_idxs` column), then pass the enriched CSV to `eval.py`:

```bash
# Step 1: Build enriched index CSV (downsamples to 8 fps, computes 2_sec_idxs column)
python src/video_index.py \
    --real-dir datasets/demo_dataset/real/ \
    --fake-dir datasets/demo_dataset/fake/ \
    --output cache/indexes/demo.csv

# Step 2: Score using the 2-second window indices
python src/eval.py \
    --csv cache/indexes/demo.csv \
    --emb-cache cache/embeddings/demo/ \
    --duration 2
```

`eval.py` auto-detects the `2_sec_idxs` column in the CSV and uses it to extract exactly 16 frames per video (2 s x 8 fps). Embeddings are cached to `--emb-cache` on first run; subsequent runs load from cache without re-running DINOv3.